# Discussion Topic: Knapsack Problem Formulation Group Discussion

You are moving from New Jersey to California and have rented a truck that can haul up to 1,100 cubic feet of furniture. The volume and value of each item you are considering moving on the truck is given below. 
- Which items should you take to California using the knapsack problem formulation?
- However, what unrealistic assumptions are you making about this real-life problem by using the knapsack problem?

| Item | Value (dollars) | Volume (cubic feet) |
| --- | --- | --- |
| Bedroom Set | 60 | 800 |
| Dining Room Set | 48 | 600 |
| Gaming Computer | 14 | 300 |
| Sofa | 31 | 400 |
| TV | 10 | 200 |

#### What is the Knapsack Problem?

It's a classic optimization problem: given a set of items with values and weights (here: volumes), and a capacity constriant, choose which items to include to maximize total value without exceeding capacity. This is a binary integer programming problem - each item is either taken (1) or left (0)

#### Understanding the Formulation
The Knapsack Problem is a type of Binary Integer Program (BIP). 

Decision Variables: 
- For each item, we create a binary variable:
$$x_i = \begin{cases} 1 & \text{if we take item } i \\ 0 & \text{if we leave it behind} \end{cases}$$

So we have 5 variables: $x_1$ (bedroom set), $x_2$ (dining room set), $x_3$ (gaming computer), $x_4$ (sofa), $x_5$ (tv)

Objective Function: 
- We want to maximize total value:
$$\text{Maximize Z } = 60x_1 + 48x_2 + 14x_3 + 31x_4 + 10x_5$$

Constraints: 
- Total volume can't exceed the truck:
$$800x_1 + 600x_2 + 300x_3 + 400x_4 + 200x_5 \le 1100$$

Binary Constraint: 
$$x_i \in \{0, 1\} \quad \forall i$$

#### General Notes

You can't load half a sofa onto a truck, each item is an all-or-nothing decision. And since our variables can only be 0 or 1 (not 0.5 or 0.3), this is specifically called a Binary Integer Program (BIP) - the most restrictive form of integer programming. 

This is also why these problems are harder to solve than regular LPs. With continuous variables, solvers can use elegant calculus-based methods. With binary variables, the solver has to be smarter - it uses a technique called Branch and Bound, which systematically explores combinations without checking every single one. 

In [7]:
# Truck capacity (cubic feet)
capacity = 1100

# Items with their value and volume
items = {
    'Bedroom Set': {'value': 60, 'volume': 800},
    'Dining Room Set': {'value': 48, 'volume': 600},
    'Gaming Computer': {'value': 14, 'volume': 300},
    'Sofa': {'value': 31, 'volume': 400},
    'TV': {'value': 10, 'volume': 200},
}

print(f"Truck capacity: {capacity} cubic feet")
print(f"Number of items to consider: {len(items)}")
print(f"Total volume of all items: {sum(v['volume'] for v in items.values())} cubic feet")
print(f"Total value of all items: ${sum(v['value'] for v in items.values())}")

Truck capacity: 1100 cubic feet
Number of items to consider: 5
Total volume of all items: 2300 cubic feet
Total value of all items: $163


In [17]:
print(f"{'Item':<20} {'Value':>8} {'Volume':>8} {'Value/CuFt':>12}")
print("-" * 52)

densities = []
for name, props in items.items():
    density = props['value'] / props['volume']
    densities.append((name, props['value'], props['volume'], density))

densities.sort(key=lambda x: x[3], reverse=True)

for name, value, volume, density in densities:
    print(f"{name:<20} ${value:>6} {volume:>6} ft^3 {density:8.4f}")

print()
print(" If we used a greedy approach (take highest density first), what would we pick?")
print(" Would that be feasible? Would it be optimal?")

Item                    Value   Volume   Value/CuFt
----------------------------------------------------
Dining Room Set      $    48    600 ft^3   0.0800
Sofa                 $    31    400 ft^3   0.0775
Bedroom Set          $    60    800 ft^3   0.0750
TV                   $    10    200 ft^3   0.0500
Gaming Computer      $    14    300 ft^3   0.0467

 If we used a greedy approach (take highest density first), what would we pick?
 Would that be feasible? Would it be optimal?


In [19]:
import pulp

prob = pulp.LpProblem("Knapsack_Moving_Problem", pulp.LpMaximize)

print(prob)

Knapsack_Moving_Problem:
MAXIMIZE
None
VARIABLES



In [21]:
x = {}
for name in items: 
    x[name] = pulp.LpVariable(name.replace(' ', '_'), cat='Binary')

print("Decision variables created:")
for name, var in x.items():
    print(f" {var.name} ∈ {{0, 1}}")

Decision variables created:
 Bedroom_Set ∈ {0, 1}
 Dining_Room_Set ∈ {0, 1}
 Gaming_Computer ∈ {0, 1}
 Sofa ∈ {0, 1}
 TV ∈ {0, 1}


In [23]:
prob += pulp.lpSum(
    items[name]['value'] * x[name] for name in items
), "Total_Value"

prob += pulp.lpSum(
    items[name]['volume'] * x[name] for name in items
) <= capacity, "Volume_Constraint"

print(prob)

Knapsack_Moving_Problem:
MAXIMIZE
60*Bedroom_Set + 48*Dining_Room_Set + 14*Gaming_Computer + 31*Sofa + 10*TV + 0.0
SUBJECT TO
Volume_Constraint: 800 Bedroom_Set + 600 Dining_Room_Set + 300 Gaming_Computer
 + 400 Sofa + 200 TV <= 1100

VARIABLES
0 <= Bedroom_Set <= 1 Integer
0 <= Dining_Room_Set <= 1 Integer
0 <= Gaming_Computer <= 1 Integer
0 <= Sofa <= 1 Integer
0 <= TV <= 1 Integer



In [25]:
status = prob.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"Solver Status: {pulp.LpStatus[prob.status]}")

Solver Status: Optimal


| Status | Meaning |
| --- | --- |
| Optimal | A single best solution was found |
| Infeasible | No feasible solution exists at all - the constraints are impossible to satisfy simultaneously |
| Unbounded | The objective can grow forever - usually means a constraint is missing |
| Undefined | Solver didn't finish - maybe it ran out of time or hit an error |

In [28]:
print("=" * 55)
print(" OPTIMAL PACKING SOLUTION")
print("=" * 55)

total_value = 0
total_volume = 0

print(f"\n{'Item':<22} {'Take?':>6} {'Value':>7} {'Volume':>9}")
print("-" * 50)

for name in items: 
    decision = int(pulp.value(x[name]))
    take = "YES" if decision == 1 else "NO"
    value = items[name]['value'] * decision
    volume = items[name]['volume'] * decision
    total_value += value
    total_volume += volume
    print(f"{name:<22} {take:>6} ${items[name]['value']:>4} {items[name]['volume']:>6} ft^3")

print("-" * 50)
print(f"{'TOTALS':<22} ${total_value:>4} {total_volume:>6} ft^3")
print()
print(f"Truck capacity used: {total_volume}/{capacity} ft^3 ({100*total_volume/capacity:.1f}%)")
print(f"Maximum value achieved: ${total_value}")
print(f"Remaining unused capacity: {capacity - total_volume} ft^3")

 OPTIMAL PACKING SOLUTION

Item                    Take?   Value    Volume
--------------------------------------------------
Bedroom Set                NO $  60    800 ft^3
Dining Room Set           YES $  48    600 ft^3
Gaming Computer            NO $  14    300 ft^3
Sofa                      YES $  31    400 ft^3
TV                         NO $  10    200 ft^3
--------------------------------------------------
TOTALS                 $  79   1000 ft^3

Truck capacity used: 1000/1100 ft^3 (90.9%)
Maximum value achieved: $79
Remaining unused capacity: 100 ft^3


In [36]:
from itertools import product as iterproduct

item_names = list(items.keys())
best_value = 0
best_combo = None
all_feasible = []

# Try all 2^5 = 32 combinations
for combo in iterproduct([0, 1], repeat=len(item_names)):
    total_vol = sum(combo[i] * items[item_names[i]]['volume'] for i in range(len(item_names)))
    total_val = sum(combo[i] * items[item_names[i]]['value'] for i in range(len(item_names)))

    if total_vol <= capacity: # feasible solution
        all_feasible.append((total_val, total_vol, combo))
        if total_val > best_value: 
            best_value = total_val
            best_combo = combo

# Sort and show top 5 feasbile solutions
all_feasible.sort(reverse=True)

print(f"Total combinations checked: {2**len(item_names)}")
print(f"Feasible solutions found: {len(all_feasible)}")
print()
print("Top 5 feasible combinations:")
print(f"{'Rank':<6} {'Value':>7} {'Volume':>8}   Items Taken")
print("-" * 65)

for rank, (val, vol, combo) in enumerate(all_feasible[:5], 1):
    taken = [item_names[i] for i in range(len(item_names)) if combo[i] == 1]
    print(f"#{rank:<5} ${val:>5}   {vol:>5} ft³   {', '.join(taken) if taken else '(nothing)'}")

print()
print(f"✅ Brute-force confirms optimal value = ${best_value}")

Total combinations checked: 32
Feasible solutions found: 16

Top 5 feasible combinations:
Rank     Value   Volume   Items Taken
-----------------------------------------------------------------
#1     $   79    1000 ft³   Dining Room Set, Sofa
#2     $   74    1100 ft³   Bedroom Set, Gaming Computer
#3     $   72    1100 ft³   Dining Room Set, Gaming Computer, TV
#4     $   70    1000 ft³   Bedroom Set, TV
#5     $   62     900 ft³   Dining Room Set, Gaming Computer

✅ Brute-force confirms optimal value = $79


### Critical Assumptions

1. Items are treated as indivisible wholes - in reality, a dining room set has a table, chairs, a hutch. A bedroom set has a bed frame, dresser, nightstands. You could choose to take some pieces and leave others. The model forces a binary all-or-nothing decision on each set, which is unrealistic
2. Volume adds perfectly like liquid - this is a big one. Real furniture has awkward shapes - a sofa has armrests and cushions, a dresser has legs. Packing a truck is actually a 3D spacial problem (known as a Bin Packing Problem), far more complex than simply summing volumes.
3. Weight is ignored - trucks have weight limits too. A solid wood bedroom set might be extremely heavy veven if it fits volumetrically.
4. Value is treated as fixed and objective - sentimental value, replacement cost in California vs. New Jersey, and item condition all affect real-wrod value
5. No cost to leaving items behind - disposing of, selling, or storing left-behind items has real costs and efforts
6. Items are indepdendent - would you really want a dining table without its chairs?